# SalamaRecover — RAG Knowledge Base Builder

This notebook processes clinical PDF documents into vector embeddings
and stores them in Supabase pgvector. This is what makes the AI
**clinically grounded** — it answers based on official Kenya MOH
guidelines, not generic internet knowledge.

## What This Notebook Does

1. **Load** — Read clinical PDFs (Kenya Nutrition Manual, WHO guidelines, etc.)
2. **Chunk** — Split into ~500-word pieces with 50-word overlap
3. **Embed** — Convert each chunk into a 768-number vector using Google's embedding model
4. **Store** — Save chunks + embeddings in Supabase pgvector
5. **Test** — Run sample queries to verify retrieval works correctly

## Prerequisites

- Google AI Studio API key (free from aistudio.google.com)
- Supabase project with pgvector enabled (free tier)
- Clinical PDFs uploaded to this Colab session

## Cost: Ksh 0
Everything in this notebook is free — Colab, embedding model, Supabase.

---
## Step 0 — Install Dependencies

These libraries handle PDF reading, text chunking, and embedding generation.

In [ ]:
# Install required packages (run once per Colab session)
!pip install -q langchain langchain-google-genai langchain-community pypdf supabase tiktoken

---
## Step 1 — Configure API Keys

Enter your Gemini API key and Supabase credentials.
These are stored in Colab's secure secrets — never in code.

In [ ]:
import os
from google.colab import userdata

# Option A: Use Colab secrets (recommended)
# Go to the key icon in the left sidebar, add these secrets:
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    SUPABASE_URL = userdata.get('SUPABASE_URL')
    SUPABASE_SERVICE_KEY = userdata.get('SUPABASE_SERVICE_KEY')
    print('Loaded API keys from Colab secrets')
except Exception:
    # Option B: Enter manually (less secure but works)
    GEMINI_API_KEY = input('Enter your Gemini API key: ')
    SUPABASE_URL = input('Enter your Supabase URL: ')
    SUPABASE_SERVICE_KEY = input('Enter your Supabase service key: ')
    print('Loaded API keys from manual input')

os.environ['GOOGLE_API_KEY'] = GEMINI_API_KEY
print(f'Supabase URL: {SUPABASE_URL[:30]}...')
print('Keys configured successfully!')

---
## Step 2 — Initialize Services

Set up the embedding model and database connection.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from supabase import create_client

# Initialize Google embedding model (free)
# This converts text into 768-dimensional vectors
embeddings = GoogleGenerativeAIEmbeddings(
    model='models/text-embedding-004',
    google_api_key=GEMINI_API_KEY,
)

# Initialize Supabase client
supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)

# Quick test — embed a single sentence
test_embedding = embeddings.embed_query('post-surgical diet in Kenya')
print(f'Embedding model working! Vector dimension: {len(test_embedding)}')
print(f'First 5 values: {test_embedding[:5]}')

---
## Step 3 — Create the Knowledge Base Table in Supabase

Run this SQL in your Supabase SQL Editor (Dashboard → SQL Editor → New Query).
You only need to do this **once** when setting up the project.

```sql
-- Enable pgvector extension
CREATE EXTENSION IF NOT EXISTS vector;

-- Create the knowledge base table
CREATE TABLE IF NOT EXISTS knowledge_base (
    id BIGSERIAL PRIMARY KEY,
    source TEXT NOT NULL,              -- e.g. 'Kenya Nutrition Manual 2010'
    content TEXT NOT NULL,             -- the actual text chunk
    embedding VECTOR(768),             -- 768-dim embedding from text-embedding-004
    metadata JSONB DEFAULT '{}',       -- {page, category, authority}
    created_at TIMESTAMPTZ DEFAULT NOW()
);

-- Create an index for fast similarity search
CREATE INDEX IF NOT EXISTS knowledge_base_embedding_idx
ON knowledge_base
USING ivfflat (embedding vector_cosine_ops)
WITH (lists = 100);

-- Create the similarity search function
-- This is what the RAG service calls to find relevant chunks
CREATE OR REPLACE FUNCTION match_knowledge_base(
    query_embedding VECTOR(768),
    match_threshold FLOAT DEFAULT 0.7,
    match_count INT DEFAULT 5
)
RETURNS TABLE (
    id BIGINT,
    source TEXT,
    content TEXT,
    metadata JSONB,
    similarity FLOAT
)
LANGUAGE plpgsql
AS $$
BEGIN
    RETURN QUERY
    SELECT
        kb.id,
        kb.source,
        kb.content,
        kb.metadata,
        1 - (kb.embedding <=> query_embedding) AS similarity
    FROM knowledge_base kb
    WHERE 1 - (kb.embedding <=> query_embedding) > match_threshold
    ORDER BY kb.embedding <=> query_embedding
    LIMIT match_count;
END;
$$;
```

After running this SQL, come back here and continue.

In [ ]:
# Verify the table exists
result = supabase.table('knowledge_base').select('id').limit(1).execute()
print(f'knowledge_base table is ready! Current rows: {len(result.data)}')

---
## Step 4 — Upload Your PDF Documents

Upload your clinical PDFs to this Colab session.
These are the documents that will power the AI's knowledge.

**Where to find these documents:**
- Kenya Nutrition Manual → MOH Kenya website
- WHO Surgical Safety Checklist → WHO website (free)
- ESPEN Guidelines → ESPEN website (free)
- Research papers → PubMed, Google Scholar, ResearchGate (free)

**In your local project, store PDFs in:** `ml/data/knowledge_base/`

In [ ]:
from google.colab import files
import os

# Create a folder for uploaded PDFs
os.makedirs('knowledge_base_pdfs', exist_ok=True)

# Upload PDFs — a file picker dialog will appear
print('Select your clinical PDF documents to upload...')
print('(Kenya Nutrition Manual, WHO guidelines, research papers, etc.)')
print()
uploaded = files.upload()

# Move uploaded files to our folder
for filename in uploaded.keys():
    os.rename(filename, f'knowledge_base_pdfs/{filename}')
    print(f'  Uploaded: {filename}')

# List all PDFs ready for processing
pdf_files = [f for f in os.listdir('knowledge_base_pdfs') if f.endswith('.pdf')]
print(f'\nTotal PDFs ready: {len(pdf_files)}')
for f in pdf_files:
    size_mb = os.path.getsize(f'knowledge_base_pdfs/{f}') / (1024 * 1024)
    print(f'  {f} ({size_mb:.1f} MB)')

---
## Step 5 — Define Document Sources

For each PDF, we define metadata: source name, category, and authority.
This metadata is stored alongside each chunk so the AI can cite its sources.

**Edit the list below** to match the PDFs you uploaded.

In [ ]:
# Define your document sources
# Edit this list to match the PDFs you uploaded

DOCUMENT_SOURCES = [
    {
        'filename': 'kenya_clinical_nutrition_manual_2010.pdf',
        'source_name': 'Kenya National Clinical Nutrition & Dietetics Manual (MOH 2010)',
        'category': 'nutrition',
        'authority': 'Kenya Ministry of Health',
    },
    # Uncomment and edit these as you add more documents:
    #
    # {
    #     'filename': 'kenya_clinical_guidelines_vol3.pdf',
    #     'source_name': 'Kenya Clinical Guidelines Volume 3',
    #     'category': 'clinical',
    #     'authority': 'Kenya Ministry of Health',
    # },
    # {
    #     'filename': 'who_surgical_safety_checklist.pdf',
    #     'source_name': 'WHO Surgical Safety Checklist',
    #     'category': 'surgical_safety',
    #     'authority': 'World Health Organization',
    # },
    # {
    #     'filename': 'espen_surgical_nutrition.pdf',
    #     'source_name': 'ESPEN Guidelines on Perioperative Nutrition',
    #     'category': 'nutrition',
    #     'authority': 'European Society for Clinical Nutrition',
    # },
]

# Validate — check that all listed files exist
for doc in DOCUMENT_SOURCES:
    path = f"knowledge_base_pdfs/{doc['filename']}"
    exists = os.path.exists(path)
    status = 'FOUND' if exists else 'NOT FOUND'
    print(f"  [{status}] {doc['filename']} — {doc['source_name']}")

---
## Step 6 — Load, Chunk, Embed, and Store

This is the main processing step. For each PDF:

1. **Load** — Extract text from every page
2. **Chunk** — Split into ~500-word pieces with 50-word overlap
3. **Embed** — Convert each chunk into a vector using Google's embedding model
4. **Store** — Insert chunk + embedding + metadata into Supabase

For a 270-page document, this takes about 5-10 minutes.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import time

# Configure the text splitter
# 500 characters per chunk, 50 character overlap
# Overlap ensures no sentence gets cut in half
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=['\n\n', '\n', '. ', ' ', ''],  # Split at natural boundaries
)

total_chunks_stored = 0

for doc_config in DOCUMENT_SOURCES:
    filepath = f"knowledge_base_pdfs/{doc_config['filename']}"

    if not os.path.exists(filepath):
        print(f"SKIPPING: {doc_config['filename']} — file not found")
        continue

    print(f"\n{'='*60}")
    print(f"Processing: {doc_config['source_name']}")
    print(f"{'='*60}")

    # Step 1: Load PDF
    print(f"  Loading PDF...")
    loader = PyPDFLoader(filepath)
    pages = loader.load()
    print(f"  Loaded {len(pages)} pages")

    # Step 2: Chunk
    print(f"  Chunking text...")
    chunks = splitter.split_documents(pages)
    print(f"  Created {len(chunks)} chunks")

    # Show a sample chunk so you can verify quality
    if chunks:
        sample = chunks[len(chunks)//2]  # Pick a chunk from the middle
        print(f"\n  Sample chunk (from page {sample.metadata.get('page', '?')}):")
        print(f"  \"{sample.page_content[:200]}...\"")
        print()

    # Step 3 & 4: Embed and Store
    print(f"  Embedding and storing chunks...")
    stored = 0
    errors = 0
    start_time = time.time()

    for i, chunk in enumerate(chunks):
        try:
            # Generate embedding for this chunk
            embedding = embeddings.embed_query(chunk.page_content)

            # Store in Supabase
            supabase.table('knowledge_base').insert({
                'source': doc_config['source_name'],
                'content': chunk.page_content,
                'embedding': embedding,
                'metadata': {
                    'page': chunk.metadata.get('page', 0),
                    'category': doc_config['category'],
                    'authority': doc_config['authority'],
                    'filename': doc_config['filename'],
                },
            }).execute()

            stored += 1

            # Progress update every 50 chunks
            if (i + 1) % 50 == 0:
                elapsed = time.time() - start_time
                rate = stored / elapsed
                remaining = (len(chunks) - i - 1) / rate if rate > 0 else 0
                print(f"    Stored {stored}/{len(chunks)} chunks "
                      f"({elapsed:.0f}s elapsed, ~{remaining:.0f}s remaining)")

            # Small delay to stay within free tier rate limits
            if (i + 1) % 10 == 0:
                time.sleep(0.5)

        except Exception as e:
            errors += 1
            if errors <= 3:  # Only show first 3 errors
                print(f"    ERROR on chunk {i}: {e}")

    elapsed = time.time() - start_time
    print(f"\n  DONE: {stored} chunks stored, {errors} errors, {elapsed:.0f}s total")
    total_chunks_stored += stored

print(f"\n{'='*60}")
print(f"COMPLETE: {total_chunks_stored} total chunks in knowledge base")
print(f"{'='*60}")

---
## Step 7 — Verify the Knowledge Base

Check how many chunks are stored and see a summary by source.

In [ ]:
# Count total chunks in the knowledge base
result = supabase.table('knowledge_base').select('id, source', count='exact').execute()
print(f'Total chunks in knowledge base: {result.count}')

# Count by source
all_records = supabase.table('knowledge_base').select('source').execute()
source_counts = {}
for record in all_records.data:
    src = record['source']
    source_counts[src] = source_counts.get(src, 0) + 1

print(f'\nChunks by source:')
for source, count in sorted(source_counts.items()):
    print(f'  {source}: {count} chunks')

---
## Step 8 — Test Retrieval (The Most Important Step!)

Now we test: when a patient asks a question, does the RAG system
retrieve the **right** chunks from the knowledge base?

For each test query, we show:
- Which chunks were retrieved
- From which document and page
- The similarity score (higher = more relevant)

**You should visually verify** that the retrieved chunks actually
answer the question. If they don't, the chunk size or overlap
may need adjusting.

In [ ]:
def test_retrieval(query: str, top_k: int = 5):
    """Test RAG retrieval for a given query."""
    print(f'\nQuery: "{query}"')
    print('-' * 60)

    # Embed the query
    query_embedding = embeddings.embed_query(query)

    # Search pgvector
    result = supabase.rpc('match_knowledge_base', {
        'query_embedding': query_embedding,
        'match_threshold': 0.5,  # Lower threshold for testing
        'match_count': top_k,
    }).execute()

    if not result.data:
        print('  No results found. Try a different query or lower the threshold.')
        return

    for i, chunk in enumerate(result.data):
        similarity = chunk.get('similarity', 0)
        source = chunk.get('source', 'Unknown')
        page = chunk.get('metadata', {}).get('page', '?')
        content = chunk.get('content', '')[:200]  # First 200 chars

        print(f'\n  Result {i+1} (similarity: {similarity:.3f})')
        print(f'  Source: {source}, Page {page}')
        print(f'  Content: "{content}..."')

    print()

In [ ]:
# Test 1: Post-caesarean diet (should find pages about soft diet progression)
test_retrieval('What should a patient eat on Day 5 after a caesarean section?')

In [ ]:
# Test 2: Clear liquid diet (should find page 66)
test_retrieval('Clear liquid diet for post-surgical patients')

In [ ]:
# Test 3: High protein diet for wound healing (should find page 75)
test_retrieval('High protein high calorie diet for wound healing after surgery')

In [ ]:
# Test 4: Fat restricted diet (should find page 74 — cholecystectomy)
test_retrieval('What diet restrictions after gallbladder removal surgery?')

In [ ]:
# Test 5: Food allergies (should find page 77)
test_retrieval('Food allergy exclusions for surgical patients with egg allergy')

In [ ]:
# Test 6: Kiswahili query — should still find relevant chunks
test_retrieval('Mgonjwa ale nini baada ya upasuaji wa caesarean siku ya tano?')

In [ ]:
# Test 7: Constipation prevention (should find page 71 — high fibre)
test_retrieval('How to prevent constipation after hernia repair surgery')

In [ ]:
# Test 8: Your own custom query — edit this!
test_retrieval('YOUR QUERY HERE')

---
## Step 9 — Test Full RAG + Gemini Response

Now let's test the complete pipeline: query → RAG retrieval → Gemini response.
This simulates exactly what happens when a patient asks a question in the app.

In [ ]:
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel('gemini-2.0-flash')

def test_full_rag(query: str, surgery_type: str = 'Caesarean Section',
                  days_since_surgery: int = 5):
    """Test the full RAG pipeline: retrieve chunks + generate response."""
    print(f'\nPatient asks: "{query}"')
    print(f'Context: {surgery_type}, Day {days_since_surgery}')
    print('=' * 60)

    # Step 1: Retrieve relevant chunks
    query_embedding = embeddings.embed_query(query)
    result = supabase.rpc('match_knowledge_base', {
        'query_embedding': query_embedding,
        'match_threshold': 0.6,
        'match_count': 5,
    }).execute()

    chunks = result.data if result.data else []
    print(f'\nRAG retrieved {len(chunks)} chunks')
    for i, chunk in enumerate(chunks):
        page = chunk.get('metadata', {}).get('page', '?')
        print(f'  Chunk {i+1}: Page {page} (similarity: {chunk.get("similarity", 0):.3f})')

    # Step 2: Build context for Gemini
    rag_context = '\n\n'.join(
        f"[Source: {c.get('source', 'Unknown')}, Page {c.get('metadata', {}).get('page', '?')}]\n"
        f"{c.get('content', '')}"
        for c in chunks
    )

    # Step 3: Generate response
    prompt = f"""You are SalamaRecover AI, a surgical recovery assistant for Kenya.
Respond based ONLY on the clinical context below. Use Kenya-local food names.
Reference page numbers. Be warm and concise.

Patient: {surgery_type}, Day {days_since_surgery}

Clinical context:
{rag_context}

Patient asks: {query}"""

    response = gemini.generate_content(prompt)

    print(f'\nAI Response:')
    print(f'{response.text}')
    print('=' * 60)

In [ ]:
# Full test 1: What to eat on Day 5 after C-section
test_full_rag(
    'What should I eat today?',
    surgery_type='Caesarean Section',
    days_since_surgery=5
)

In [ ]:
# Full test 2: Diet after gallbladder surgery
test_full_rag(
    'What foods should I avoid?',
    surgery_type='Cholecystectomy',
    days_since_surgery=3
)

In [ ]:
# Full test 3: Kiswahili question
test_full_rag(
    'Nile nini leo? Nina njaa sana.',
    surgery_type='Appendectomy',
    days_since_surgery=7
)

---
## Step 10 — Adding New Documents Later

When you find new clinical documents (research papers, WHO guidelines, etc.),
come back to this notebook and:

1. Upload the new PDF (Step 4)
2. Add it to `DOCUMENT_SOURCES` (Step 5)
3. Re-run Steps 6-8

The new document's knowledge will be immediately available to the AI.
You don't need to retrain anything — just add chunks to the database.

### Useful Sources to Add Over Time

| Document | Where to Find It |
|---|---|
| Kenya Clinical Guidelines Vol 3 | MOH Kenya website |
| WHO Surgical Safety Checklist | who.int (free PDF) |
| ESPEN Perioperative Nutrition | espen.org (free) |
| KNH Post-Op Protocols | KNH medical library |
| Research papers | PubMed, Google Scholar |
| Kenya Essential Medicines List | MOH Kenya |

### To Clear and Rebuild the Knowledge Base

If you need to start fresh (e.g., chunk size was wrong):

```sql
-- Run in Supabase SQL Editor
TRUNCATE TABLE knowledge_base;
```

Then re-run this notebook from Step 6.

In [ ]:
print('Notebook complete!')
print()
print('Your RAG knowledge base is ready.')
print('The AI will now answer based on Kenya MOH clinical guidelines.')
print()
print('Next notebook: 02_gemini_clinical_prompts.ipynb')
print('  → Design and test the clinical prompts for risk scoring, diet, and chat')